# Model-Agnostic Per-Stimulus Metrics Pipeline

Run all cells top-to-bottom to compute per-stimulus NLP metrics for any supported
HuggingFace causal LM. **Only the config cell below needs editing.**

Output: `outputs/{model_prefix}_metrics.csv` with columns:
- `stim_id`, `dataset`, `critical_word`, `n_tokens`
- `last_layer_surprisal`
- `layer_00` … `layer_NN` (tuned-lens layer-wise surprisal, if lens is available)
- `lang_cosine_dist` (cosine distance of language-network vectors at pre/critical token)
- `lang_mae_update` (mean absolute difference of language-network vectors at pre/critical token; analogous to SGM gestalt update)
- `shallow_surprisal` (Li & Futrell 2024: D_KL[λ‖p₀], λ=SHALLOW_LAMBDA)


In [1]:
# ── CONFIG — the only cell you need to edit ──────────────────────────────────
MODEL_NAME = 'Qwen/Qwen3-0.6B'  # any HuggingFace causal LM ID


In [77]:
import gc
import os
# Redirect HF cache to project storage (avoids home-directory quota)
os.environ['HF_HOME'] = '/orcd/data/evelina9/001/USERS/samhutch/.huggingface_cache'
os.makedirs(os.environ['HF_HOME'], exist_ok=True)

import re
import math
import editdistance
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from scipy import stats
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer

# ── Pipeline constants ────────────────────────────────────────────────────────
MANIFEST_PATH    = 'stims_for_modeling.csv'
N400_PATH        = 'combined_clean_n400.csv'
OUTPUT_DIR       = 'outputs'
CHECKPOINT_EVERY = 250
LOCALIZER_N      = 100
LOCALIZER_SEED   = 42

# Li & Futrell (2024) shallow surprisal: D_KL[p_λ ‖ p_0]
# γ=8 is fixed in the paper; λ tunes processing depth (paper varies per dataset; 1.0 is a neutral default)
SHALLOW_LAMBDA = 1.0
SHALLOW_GAMMA  = 8.0
SHALLOW_TOP_K  = 200   # vocabulary candidates sampled by p_0 mass

# Models with pre-trained tuned-lens probes on AlignmentResearch/tuned-lens Hub space
# Source: https://huggingface.co/spaces/AlignmentResearch/tuned-lens/tree/main/lens
TUNED_LENS_SUPPORTED = {
    'gpt2', 'gpt2-large', 'gpt2-xl',
    'CarperAI/stable-vicuna-13b',
    'EleutherAI/gpt-neox-20b',
    'EleutherAI/pythia-70m-deduped',   'EleutherAI/pythia-70m-deduped-v0',
    'EleutherAI/pythia-160m-deduped',  'EleutherAI/pythia-160m-deduped-v0',
    'EleutherAI/pythia-410m-deduped',  'EleutherAI/pythia-410m-deduped-v0',
    'EleutherAI/pythia-1b-deduped-v0',
    'EleutherAI/pythia-1.4b-deduped',  'EleutherAI/pythia-1.4b-deduped-v0',
    'EleutherAI/pythia-2.8b-deduped',  'EleutherAI/pythia-2.8b-deduped-v0',
    'EleutherAI/pythia-6.9b-deduped',  'EleutherAI/pythia-6.9b-deduped-v0',
    'EleutherAI/pythia-12b-deduped',   'EleutherAI/pythia-12b-deduped-v0',
    'facebook/opt-125m', 'facebook/opt-1.3b', 'facebook/opt-6.7b',
    'lmsys/vicuna-13b-v1.1',
    'meta-llama/Llama-2-7b-hf',   'meta-llama/Llama-2-7b-chat-hf',
    'meta-llama/Llama-2-13b-hf',  'meta-llama/Llama-2-13b-chat-hf',
    'meta-llama/Meta-Llama-3-8B', 'meta-llama/Meta-Llama-3-8B-Instruct',
}

def derive_model_prefix(model_name):
    return model_name.replace('/', '_').replace('-', '_').lower()

def check_tuned_lens_available(model_name):
    return model_name in TUNED_LENS_SUPPORTED

def free_model_memory(device_type: str) -> None:
    """Release Python + accelerator memory after deleting a model."""
    gc.collect()
    if device_type == 'cuda':
        torch.cuda.empty_cache()
    elif device_type == 'mps':
        torch.mps.empty_cache()

def load_model(model_name, torch_dtype, device):
    """Load a causal LM directly onto the target device.

    Uses device_map instead of a separate .to(device) call so weights are
    streamed from disk straight to GPU — no full CPU copy followed by a
    GPU copy, which avoids OOM when the GPU allocator's reserved pool is
    still holding memory from a previous model.
    """
    cfg = AutoConfig.from_pretrained(model_name)
    if (hasattr(cfg, 'quantization_config')
            and not hasattr(cfg.quantization_config, 'quant_method')):
        del cfg.quantization_config
    return AutoModelForCausalLM.from_pretrained(
        model_name,
        config=cfg,
        torch_dtype=torch_dtype,
        device_map=str(device),
        use_safetensors=True,
    )

MODEL_PREFIX          = derive_model_prefix(MODEL_NAME)
TUNED_LENS_AVAILABLE  = check_tuned_lens_available(MODEL_NAME)
METRICS_PATH          = os.path.join(OUTPUT_DIR, f'{MODEL_PREFIX}_metrics.csv')
LAST_LAYER_PATH       = os.path.join(OUTPUT_DIR, f'{MODEL_PREFIX}_last_layer_surprisal.csv')
LAYER_SURPRISALS_PATH = os.path.join(OUTPUT_DIR, f'{MODEL_PREFIX}_layer_surprisals.csv')
LANG_COSINE_PATH      = os.path.join(OUTPUT_DIR, f'{MODEL_PREFIX}_lang_cosine.csv')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'HF_HOME    : {os.environ["HF_HOME"]}')
print(f'Model      : {MODEL_NAME}')
print(f'Prefix     : {MODEL_PREFIX}')
lens_status = 'available' if TUNED_LENS_AVAILABLE else 'NOT available — layer/shallow/deep columns will be NaN'
print(f'Tuned-lens : {lens_status}')
print(f'Output     : {METRICS_PATH}')

HF_HOME    : /orcd/data/evelina9/001/USERS/samhutch/.huggingface_cache
Model      : Qwen/Qwen3-0.6B
Prefix     : qwen_qwen3_0.6b
Tuned-lens : NOT available — layer/shallow/deep columns will be NaN
Output     : outputs/qwen_qwen3_0.6b_metrics.csv


In [78]:
# ── Device detection ─────────────────────────────────────────────────────────
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f'Device: {device}')


def find_word_positions(sentence, critical_word, tokenizer):
    """
    Locate the BPE token positions of critical_word within sentence.

    Returns (word_positions, input_ids) where word_positions is a list of
    token indices spanning critical_word.  Returns ([], None) when the word
    only appears at position 0 (no left context) or is not found at all.

    Handles apostrophe normalisation (curly/straight apostrophes stripped to
    match cleaned stimuli) and multi-token critical words (BPE subwords).
    """
    sentence    = sentence.strip()
    search_word = re.sub(r"[\u2018\u2019\u02bc']", '', critical_word.lower())
    pattern     = r'(?<![a-zA-Z])' + re.escape(search_word) + r'(?![a-zA-Z])'

    encoding  = tokenizer(
        sentence,
        return_tensors='pt',
        add_special_tokens=False,
        return_offsets_mapping=True,
    )
    input_ids = encoding['input_ids'][0]
    offsets   = encoding['offset_mapping'][0].tolist()

    for match in re.finditer(pattern, sentence.lower()):
        char_start, char_end = match.start(), match.end()
        positions = [
            i for i, (s, e) in enumerate(offsets)
            if e > char_start and s < char_end and e > s
        ]
        if positions and positions[0] > 0:
            return positions, input_ids

    return [], None


Device: cuda


In [79]:
manifest = pd.read_csv(MANIFEST_PATH)
print(f'Manifest: {len(manifest):,} rows | {manifest["stim_id"].nunique():,} unique stimuli')
print(manifest['dataset'].value_counts().to_string())


Manifest: 2,808 rows | 2,808 unique stimuli
dataset
frank_2015        1668
ryskin_2021        640
michaelov_2024     500


---
## Pass 1 — Last-layer surprisal (float16)


In [80]:
# Free any model / lens still in memory from a previous run before loading
for _var in ('model', 'lens'):
    if _var in globals():
        del globals()[_var]
free_model_memory(device.type)

dtype = torch.float16 if device.type in ('cuda', 'mps') else torch.float32
print(f'Loading {MODEL_NAME} ({dtype})...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = load_model(MODEL_NAME, dtype, device)
model.eval()

N_LAYERS        = model.config.num_hidden_layers
N_EMBD          = model.config.hidden_size
layer_col_names = [f'layer_{i:02d}' for i in range(N_LAYERS)]
print(f'Loaded | layers={N_LAYERS} | d_model={N_EMBD} | dtype={model.dtype}')

if device.type == 'mps':
    with torch.no_grad():
        _dummy = torch.zeros(1, 16, dtype=torch.long).to(device)
        _ = model(_dummy)
    del _dummy
    torch.mps.empty_cache()
    print('MPS warmup done')

Loading Qwen/Qwen3-0.6B (torch.float16)...


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Loaded | layers=28 | d_model=1024 | dtype=torch.float16


In [81]:
LAST_LAYER_CKPT = os.path.join(OUTPUT_DIR, f'{MODEL_PREFIX}_last_layer_surprisal_ckpt.csv')

_ll_done = (
    os.path.exists(LAST_LAYER_PATH)
    and 'shallow_surprisal' in (_ll_df := pd.read_csv(LAST_LAYER_PATH)).columns
    and len(_ll_df) == len(manifest)
)
if _ll_done:
    print(f'Last-layer surprisal complete — loading {LAST_LAYER_PATH}')
    last_layer_df = _ll_df
else:
    _records, _start = [], 0
    if os.path.exists(LAST_LAYER_CKPT):
        _cp = pd.read_csv(LAST_LAYER_CKPT)
        if 'shallow_surprisal' in _cp.columns:
            _records = _cp.to_dict('records')
            _start   = len(_records)
            print(f'Resuming from checkpoint: {_start}/{len(manifest)} done')
        else:
            print('Checkpoint predates shallow_surprisal — recomputing from scratch')

    # Embedding matrix on CPU in float32 for distortion distance (avoids device-mismatch)
    embed_weight = model.get_input_embeddings().weight.detach().cpu().float()

    n = len(manifest)
    for i, (_, row) in enumerate(manifest.iloc[_start:].iterrows(), start=_start):
        word_positions, input_ids = find_word_positions(row['stim'], row['critical_word'], tokenizer)
        if not word_positions:
            _records.append({
                'stim_id': row['stim_id'], 'dataset': row['dataset'],
                'critical_word': row['critical_word'],
                'last_layer_surprisal': np.nan, 'n_tokens': 0,
                'shallow_surprisal': np.nan,
            })
        else:
            prefix_ids = input_ids[:word_positions[-1] + 1].unsqueeze(0).to(device)
            with torch.no_grad():
                logits    = model(prefix_ids).logits[0]
                log_probs = F.log_softmax(logits, dim=-1)

            surprisal_nats = sum(
                -log_probs[pos - 1, input_ids[pos].item()].item() for pos in word_positions
            )

            # ── Li & Futrell (2024) shallow surprisal: D_KL[p_λ ‖ p_0] ──────────────────
            # p_0 = LM distribution before the critical word's first token
            # p_λ(w) ∝ p_0(w) · exp(-λ · (Levenshtein(w,x) + γ · cosine(embed_w, embed_x)))
            crit_first = word_positions[0]
            if crit_first > 0:
                log_p0      = log_probs[crit_first - 1, :].cpu().float()  # (vocab,)
                top_k_lp, top_k_ids_t = torch.topk(log_p0, SHALLOW_TOP_K)
                top_k_lp_np  = top_k_lp.numpy()
                top_k_ids_np = top_k_ids_t.numpy()

                cand_embeds  = embed_weight[top_k_ids_t].numpy()                      # (K, d)
                actual_embed = embed_weight[input_ids[word_positions]].numpy().mean(0) # (d,)
                actual_norm  = np.linalg.norm(actual_embed)
                actual_str   = tokenizer.decode(input_ids[word_positions].tolist()).strip().lower()

                dists = np.empty(SHALLOW_TOP_K)
                for j in range(SHALLOW_TOP_K):
                    cand_str = tokenizer.decode([int(top_k_ids_np[j])]).strip().lower()
                    d_phi    = editdistance.eval(cand_str, actual_str)
                    cn       = np.linalg.norm(cand_embeds[j])
                    d_sigma  = (1.0 - float(np.dot(cand_embeds[j], actual_embed) / (cn * actual_norm))
                                if cn > 0 and actual_norm > 0 else 1.0)
                    dists[j] = d_phi + SHALLOW_GAMMA * d_sigma

                log_pw_un   = top_k_lp_np - SHALLOW_LAMBDA * dists
                lse         = np.log(np.sum(np.exp(log_pw_un - log_pw_un.max()))) + log_pw_un.max()
                log_pw      = log_pw_un - lse
                shallow_sup = float(np.sum(np.exp(log_pw) * (log_pw - top_k_lp_np)))
            else:
                shallow_sup = np.nan

            _records.append({
                'stim_id': row['stim_id'], 'dataset': row['dataset'],
                'critical_word': row['critical_word'],
                'last_layer_surprisal': surprisal_nats / math.log(2),
                'n_tokens': len(word_positions),
                'shallow_surprisal': shallow_sup,
            })

        if device.type == 'mps' and i % 50 == 0:
            torch.mps.empty_cache()
        if i % 100 == 0:
            print(f'{i}/{n}', flush=True)
        if (i + 1) % CHECKPOINT_EVERY == 0:
            pd.DataFrame(_records).to_csv(LAST_LAYER_CKPT, index=False)

    last_layer_df = pd.DataFrame(_records)
    last_layer_df.to_csv(LAST_LAYER_PATH, index=False)
    nan_n = last_layer_df['last_layer_surprisal'].isna().sum()
    print(f'Saved {LAST_LAYER_PATH}: {last_layer_df.shape} | NaN rows: {nan_n}')


0/2808
100/2808
200/2808
300/2808
400/2808
500/2808
600/2808
700/2808
800/2808
900/2808
1000/2808
1100/2808
1200/2808
1300/2808
1400/2808
1500/2808
1600/2808
1700/2808
1800/2808
1900/2808
2000/2808
2100/2808
2200/2808
2300/2808
2400/2808
2500/2808
2600/2808
2700/2808
2800/2808
Saved outputs/qwen_qwen3_0.6b_last_layer_surprisal.csv: (2808, 6) | NaN rows: 0


---
## Pass 2 — Layer-wise surprisal via tuned-lens (float32, conditional)

Skipped automatically if no pre-trained lens is available for `MODEL_NAME`.
Layer-wise and shallow/deep columns will be NaN in the master CSV.


In [82]:
if TUNED_LENS_AVAILABLE:
    from tuned_lens.nn.lenses import TunedLens

    if model.dtype != torch.float32:
        print(f'Reloading {MODEL_NAME} {model.dtype} → float32 for tuned-lens...')
        del model
        free_model_memory(device.type)
        model = load_model(MODEL_NAME, torch.float32, device)
        model.eval()
        print(f'Reloaded | dtype={model.dtype}')
    else:
        print('Model already float32 — no reload needed')

    print('Loading tuned-lens probes...')
    lens = TunedLens.from_model_and_pretrained(model)
    lens = lens.to(device)
    lens.eval()
    print(f'Tuned-lens loaded: {len(lens)} probes')
else:
    print(f'Tuned-lens not available for {MODEL_NAME} — Pass 2 skipped.')
    BEST_N400_LAYER = N_LAYERS - 1
    print(f'BEST_N400_LAYER fallback: {BEST_N400_LAYER} (final layer)')

Tuned-lens not available for Qwen/Qwen3-0.6B — Pass 2 skipped.
BEST_N400_LAYER fallback: 27 (final layer)


In [83]:
if TUNED_LENS_AVAILABLE:
    LAYER_SURPRISALS_CKPT = os.path.join(OUTPUT_DIR, f'{MODEL_PREFIX}_layer_surprisals_ckpt.csv')

    if os.path.exists(LAYER_SURPRISALS_PATH) and len(pd.read_csv(LAYER_SURPRISALS_PATH)) == len(manifest):
        print(f'Layer surprisals complete — loading {LAYER_SURPRISALS_PATH}')
        layer_df = pd.read_csv(LAYER_SURPRISALS_PATH)
    else:
        _records_ls, _start_ls = [], 0
        if os.path.exists(LAYER_SURPRISALS_CKPT):
            _cp        = pd.read_csv(LAYER_SURPRISALS_CKPT)
            _records_ls = _cp.to_dict('records')
            _start_ls   = len(_records_ls)
            print(f'Resuming from checkpoint: {_start_ls}/{len(manifest)} done')

        n = len(manifest)
        for i, (_, row) in enumerate(manifest.iloc[_start_ls:].iterrows(), start=_start_ls):
            word_positions, input_ids = find_word_positions(row['stim'], row['critical_word'], tokenizer)
            if not word_positions:
                rec = {
                    'stim_id': row['stim_id'], 'dataset': row['dataset'],
                    'critical_word': row['critical_word'], 'n_tokens': 0,
                }
                rec.update(dict(zip(layer_col_names, [np.nan] * N_LAYERS)))
                _records_ls.append(rec)
            else:
                prefix_ids = input_ids[:word_positions[-1] + 1].unsqueeze(0).to(device)
                with torch.no_grad():
                    out = model(prefix_ids, output_hidden_states=True)

                layer_sups = []
                for layer_idx in range(N_LAYERS):
                    # hidden_states[0]=embedding, [1..N]=transformer block outputs
                    h = out.hidden_states[layer_idx + 1]           # (1, seq, d_model)
                    with torch.no_grad():
                        logits    = lens.forward(h, layer_idx)     # (1, seq, vocab)
                        log_probs = F.log_softmax(logits[0], dim=-1)  # (seq, vocab)
                    surprisal_nats = sum(
                        -log_probs[pos - 1, input_ids[pos].item()].item() for pos in word_positions
                    )
                    layer_sups.append(surprisal_nats / math.log(2))

                rec = {
                    'stim_id': row['stim_id'], 'dataset': row['dataset'],
                    'critical_word': row['critical_word'], 'n_tokens': len(word_positions),
                }
                rec.update(dict(zip(layer_col_names, layer_sups)))
                _records_ls.append(rec)

            if device.type == 'mps' and i % 50 == 0:
                torch.mps.empty_cache()
            if i % 100 == 0:
                print(f'{i}/{n}', flush=True)
            if (i + 1) % CHECKPOINT_EVERY == 0:
                pd.DataFrame(_records_ls).to_csv(LAYER_SURPRISALS_CKPT, index=False)

        layer_df = pd.DataFrame(_records_ls)
        layer_df.to_csv(LAYER_SURPRISALS_PATH, index=False)
        print(f'Saved {LAYER_SURPRISALS_PATH}: {layer_df.shape}')

    # Determine BEST_N400_LAYER: layer maximally correlated with frank_2015 N400
    # Aggregate N400 to one row per stimulus to avoid pseudo-replication
    _n400_agg = (
        pd.read_csv(N400_PATH)
        .groupby(['stim_id', 'critical_word'])['meanAmp_z']
        .mean()
        .reset_index()
    )
    _merged = (
        layer_df
        .merge(_n400_agg, on=['stim_id', 'critical_word'])
        .dropna(subset=layer_col_names + ['meanAmp_z'])
    )
    _frank = _merged[_merged['dataset'] == 'frank_2015']
    _r_by_layer = [_frank[col].corr(_frank['meanAmp_z']) for col in layer_col_names]
    if len(_frank) == 0 or all(v != v for v in _r_by_layer):  # all-NaN guard
        import warnings
        warnings.warn('No frank_2015 rows survived merge/dropna — falling back to N_LAYERS-1. '
                      'Check that N400_PATH stim_ids match layer_df.')
        BEST_N400_LAYER = N_LAYERS - 1
    else:
        BEST_N400_LAYER = int(np.argmax(np.abs(_r_by_layer)))
    print(f'BEST_N400_LAYER = {BEST_N400_LAYER}  '
          f'(r={_r_by_layer[BEST_N400_LAYER]:.3f} with frank_2015 N400)')


---
## Pass 3 — Language-network localizer & cosine distance (float32)


In [84]:
# Ensure float32 for language-network passes regardless of which Pass 2 path was taken.
# (Unconditional dtype check — a no-op when Pass 2 already reloaded float32.)
if model.dtype != torch.float32:
    print(f'Reloading {MODEL_NAME} {model.dtype} → float32 for language-network pass...')
    del model
    free_model_memory(device.type)
    model = load_model(MODEL_NAME, torch.float32, device)
    model.eval()
    print(f'Reloaded | dtype={model.dtype}')
else:
    print('Model already float32')

Reloading Qwen/Qwen3-0.6B torch.float16 → float32 for language-network pass...


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Reloaded | dtype=torch.float32


In [85]:
LANG_NET_PATH = os.path.join(OUTPUT_DIR, f'{MODEL_PREFIX}_language_network_units.csv')

_sent_file    = 'data/fedorenko_localizer_sentences.txt'
_nonword_file = 'data/fedorenko_localizer_nonwords.txt'
if not os.path.exists(_sent_file) or not os.path.exists(_nonword_file):
    raise FileNotFoundError(
        f'Fedorenko localizer files not found. Expected:\n'
        f'  {_sent_file}\n  {_nonword_file}\n'
        'Download from https://evlab.mit.edu/assets/pages/langloc.html'
    )

with open(_sent_file) as _fh:
    _all_sentences = _fh.read().splitlines()
with open(_nonword_file) as _fh:
    _all_nonwords = _fh.read().splitlines()

# Single RNG instance: two sequential calls avoid seed-reuse index overlap
_rng         = np.random.default_rng(LOCALIZER_SEED)
_sent_idx    = _rng.choice(len(_all_sentences), size=LOCALIZER_N, replace=False)
_nonword_idx = _rng.choice(len(_all_nonwords),  size=LOCALIZER_N, replace=False)
localizer_sentences = [_all_sentences[i] for i in sorted(_sent_idx)]
localizer_nonwords  = [_all_nonwords[i]  for i in sorted(_nonword_idx)]

print(f'Localizer: {len(localizer_sentences)} sentences / {len(localizer_nonwords)} nonwords')


def _extract_mean_activations(texts, model, tokenizer, device, n_layers):
    """Returns (n, n_layers, d_model) array of mean-pooled hidden states."""
    acts = []
    for i, text in enumerate(texts):
        enc       = tokenizer(text, return_tensors='pt', add_special_tokens=False)
        input_ids = enc['input_ids'].to(device)
        with torch.no_grad():
            out = model(input_ids, output_hidden_states=True)
        h = torch.stack(
            [out.hidden_states[l + 1].squeeze(0).mean(0) for l in range(n_layers)], dim=0
        )  # (n_layers, d_model)
        acts.append(h.cpu().float().numpy())
        if device.type == 'mps' and i % 20 == 0:
            torch.mps.empty_cache()
    return np.stack(acts, axis=0)  # (n, n_layers, d_model)


print('Extracting sentence activations...')
sent_acts    = _extract_mean_activations(localizer_sentences, model, tokenizer, device, N_LAYERS)
print('Extracting nonword activations...')
nonword_acts = _extract_mean_activations(localizer_nonwords,  model, tokenizer, device, N_LAYERS)

sent_flat    = sent_acts.reshape(LOCALIZER_N, -1)    # (100, N_LAYERS * N_EMBD)
nonword_flat = nonword_acts.reshape(LOCALIZER_N, -1)
t_stats, _   = stats.ttest_ind(sent_flat, nonword_flat, axis=0, equal_var=False)

n_lang_units        = int(0.01 * N_LAYERS * N_EMBD)
lang_unit_flat_indices = np.argsort(-t_stats)[:n_lang_units]  # top 1% most sentence-selective units

lang_net_df = pd.DataFrame({
    'layer':    lang_unit_flat_indices // N_EMBD,
    'unit_idx': lang_unit_flat_indices % N_EMBD,
    't_stat':   t_stats[lang_unit_flat_indices],
})
lang_net_df.to_csv(LANG_NET_PATH, index=False)
print(f'Saved {LANG_NET_PATH}: {len(lang_net_df)} language-network units')
print(f't_stat range: {lang_net_df["t_stat"].min():.2f} — {lang_net_df["t_stat"].max():.2f}')


Localizer: 100 sentences / 100 nonwords
Extracting sentence activations...
Extracting nonword activations...
Saved outputs/qwen_qwen3_0.6b_language_network_units.csv: 286 language-network units
t_stat range: 24.12 — 47.95


In [86]:
LANG_COSINE_CKPT = os.path.join(OUTPUT_DIR, f'{MODEL_PREFIX}_lang_cosine_ckpt.csv')

_lc_existing = pd.read_csv(LANG_COSINE_PATH) if os.path.exists(LANG_COSINE_PATH) else None
if (_lc_existing is not None
        and len(_lc_existing) == len(manifest)
        and 'lang_mae_update' in _lc_existing.columns):
    print(f'Cosine distance complete — loading {LANG_COSINE_PATH}')
    cosine_df = _lc_existing
else:
    _records_cos, _start_cos = [], 0
    if os.path.exists(LANG_COSINE_CKPT):
        _cp = pd.read_csv(LANG_COSINE_CKPT)
        if 'lang_mae_update' in _cp.columns:
            _records_cos = _cp.to_dict('records')
            _start_cos   = len(_records_cos)
            print(f'Resuming from checkpoint: {_start_cos}/{len(manifest)} done')
        else:
            print('Checkpoint predates lang_mae_update — recomputing from scratch')

    n = len(manifest)
    for i, (_, row) in enumerate(manifest.iloc[_start_cos:].iterrows(), start=_start_cos):
        word_positions, input_ids = find_word_positions(row['stim'], row['critical_word'], tokenizer)
        if not word_positions:
            dist = np.nan
            mae  = np.nan
        else:
            pre_pos   = word_positions[0] - 1   # token immediately before critical word
            crit_pos  = word_positions[-1]       # last BPE token of critical word
            prefix_ids = input_ids[:crit_pos + 1].unsqueeze(0).to(device)
            with torch.no_grad():
                out = model(prefix_ids, output_hidden_states=True)

            def _lang_vec(pos):
                h    = torch.stack(
                    [out.hidden_states[l + 1][0, pos, :] for l in range(N_LAYERS)], dim=0
                )  # (N_LAYERS, N_EMBD)
                flat = h.cpu().float().numpy().flatten()     # (N_LAYERS * N_EMBD,)
                return flat[lang_unit_flat_indices]          # (n_lang_units,)

            v_pre  = _lang_vec(pre_pos)
            v_crit = _lang_vec(crit_pos)
            norm_pre  = np.linalg.norm(v_pre)
            norm_crit = np.linalg.norm(v_crit)
            if norm_pre == 0 or norm_crit == 0:
                dist = np.nan
                mae  = np.nan
            else:
                dist = 1.0 - float(np.dot(v_pre, v_crit) / (norm_pre * norm_crit))
                mae  = float(np.mean(np.abs(v_crit - v_pre)))

        _records_cos.append({
            'stim_id': row['stim_id'], 'dataset': row['dataset'],
            'critical_word': row['critical_word'],
            'lang_cosine_dist': dist, 'lang_mae_update': mae,
        })
        if device.type == 'mps' and i % 50 == 0:
            torch.mps.empty_cache()
        if i % 100 == 0:
            print(f'{i}/{n}', flush=True)
        if (i + 1) % CHECKPOINT_EVERY == 0:
            pd.DataFrame(_records_cos).to_csv(LANG_COSINE_CKPT, index=False)

    cosine_df = pd.DataFrame(_records_cos)
    cosine_df.to_csv(LANG_COSINE_PATH, index=False)
    print(f'Saved {LANG_COSINE_PATH}: {cosine_df.shape}')
    _valid_cos = cosine_df['lang_cosine_dist'].dropna()
    _valid_mae = cosine_df['lang_mae_update'].dropna()
    print(f'lang_cosine_dist range: [{_valid_cos.min():.4f}, {_valid_cos.max():.4f}]  (expected [0, 2])')
    print(f'lang_mae_update  range: [{_valid_mae.min():.6f}, {_valid_mae.max():.6f}]')


0/2808


100/2808
200/2808
300/2808
400/2808
500/2808
600/2808
700/2808
800/2808
900/2808
1000/2808
1100/2808
1200/2808
1300/2808
1400/2808
1500/2808
1600/2808
1700/2808
1800/2808
1900/2808
2000/2808
2100/2808
2200/2808
2300/2808
2400/2808
2500/2808
2600/2808
2700/2808
2800/2808
Saved outputs/qwen_qwen3_0.6b_lang_cosine.csv: (2808, 5)
lang_cosine_dist range: [0.0008, 0.4587]  (expected [0, 2])
lang_mae_update  range: [1.064536, 733.861389]


---
## Merge — Master metrics CSV


In [87]:
# ── Load from per-measure CSVs ───────────────────────────────────────────────
_ll  = pd.read_csv(LAST_LAYER_PATH)
_lyr = pd.read_csv(LAYER_SURPRISALS_PATH) if os.path.exists(LAYER_SURPRISALS_PATH) else None
_cos = pd.read_csv(LANG_COSINE_PATH)

# ── Merge ────────────────────────────────────────────────────────────────────
metrics = manifest[['stim_id', 'dataset', 'critical_word']].copy()

_ll_cols = [c for c in _ll.columns if c not in ('stim_id', 'dataset', 'critical_word')]
metrics = metrics.merge(_ll[['stim_id', 'critical_word'] + _ll_cols],
                        on=['stim_id', 'critical_word'], how='left')

if _lyr is not None:
    _lyr_cols = [c for c in _lyr.columns if c not in ('stim_id', 'dataset', 'critical_word')]
    metrics = metrics.merge(_lyr[['stim_id', 'critical_word'] + _lyr_cols],
                            on=['stim_id', 'critical_word'], how='left')

_cos_cols = [c for c in _cos.columns if c not in ('stim_id', 'dataset', 'critical_word')]
metrics = metrics.merge(_cos[['stim_id', 'critical_word'] + _cos_cols],
                        on=['stim_id', 'critical_word'], how='left')

metrics.to_csv(METRICS_PATH, index=False)
print(f'Saved {METRICS_PATH}')
print(f'Shape  : {metrics.shape}')
print(f'Columns: {list(metrics.columns)}')
print(f'\nNaN counts:')
print(metrics.isna().sum().to_string())
assert metrics['stim_id'].nunique() == len(metrics), 'Duplicate stim_id detected in output!'
print(f'\nstim_id unique: OK ({metrics["stim_id"].nunique()} rows, no duplicates)')


Saved outputs/qwen_qwen3_0.6b_metrics.csv
Shape  : (2808, 8)
Columns: ['stim_id', 'dataset', 'critical_word', 'last_layer_surprisal', 'n_tokens', 'shallow_surprisal', 'lang_cosine_dist', 'lang_mae_update']

NaN counts:
stim_id                 0
dataset                 0
critical_word           0
last_layer_surprisal    0
n_tokens                0
shallow_surprisal       0
lang_cosine_dist        0
lang_mae_update         0

stim_id unique: OK (2808 rows, no duplicates)
